In [95]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle


In [96]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [97]:
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [98]:
## encoding categorical variables
label_encoder = LabelEncoder()
data['Gender']= label_encoder.fit_transform(data['Gender'])

In [99]:
onehot_encoder = OneHotEncoder(sparse_output=False)

geo_encoded = onehot_encoder.fit_transform(data[['Geography']])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder.get_feature_names_out(['Geography'])
)

geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [100]:
df=pd.concat([data, geo_encoded_df], axis=1).drop(['Geography'], axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [101]:
X=df.drop(['EstimatedSalary'], axis=1)
y=df['EstimatedSalary']

In [102]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [103]:
## make pickle files for the scaler and encoders
with open('scaler_salary.pkl', 'wb') as file:
    pickle.dump(scaler, file)

with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)    

with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)

In [104]:
## ANN Regression Model
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),   
    Dense(1, activation='linear')
])  

## compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)

d:\Data Science\ANN_Project\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [105]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [106]:

import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [107]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    callbacks=[early_stopping, tensorboard_callback]
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 13307904000.0000 - mean_absolute_error: 99976.8594 - val_loss: 13695188992.0000 - val_mean_absolute_error: 102024.9375
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 13229045760.0000 - mean_absolute_error: 99584.8672 - val_loss: 13532449792.0000 - val_mean_absolute_error: 101230.1406
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 12941126656.0000 - mean_absolute_error: 98145.5781 - val_loss: 13095056384.0000 - val_mean_absolute_error: 99096.6172
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 12336764928.0000 - mean_absolute_error: 95149.1094 - val_loss: 12306725888.0000 - val_mean_absolute_error: 95230.9531
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 11381117952.0000 - mean_absolute_error: 90402.5938 - val_loss: 11171140608.0000 - val_mean_absolute_error: 89643.3203
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 10122530816.0000 - mean_absolute_error: 84

In [ ]:
## Evaluate the model
model.evaluate(X_test_scaled, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3355018496.0000 - mean_absolute_error: 50106.8242


[3355018496.0, 50106.82421875]

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6009 (pid 20132), started 0:00:27 ago. (Use '!kill 20132' to kill it.)

In [ ]:
model.save('salary_regression_model.h5')